In [2]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))

OpenAI key loaded: True


## RAG 1. Install dependencies

In [3]:
!python -m pip install llama-index llama-index-llms-openai llama-index-embeddings-openai openai

zsh:1: command not found: python


## RAG 2. Configuration

In [4]:
import os

# Path to the NEURON repo docs folder (relative to doc-project/)
DOCS_PATHS = [
    "../docs/nmodl",   
    "../docs/progref",
]

# Where to save the persistent index (so you don't re-index every time)
INDEX_STORE_PATH = "neuron_index"

## Step 1. Summarize Threads

In [12]:
import json
from llama_index.llms.openai import OpenAI

SUMMARIZE_PROMPT = """
You are helping to process NEURON simulator forum threads for documentation purposes.

Read the following forum thread and extract:
1. A brief summary of the problem and solution (2-3 sentences)
2. A list of NEURON functions, methods, or classes mentioned (e.g. CVode.event, NetCon.record, fadvance)

Return ONLY a JSON object with exactly these fields:
{
  "summary": "...",
  "functions_mentioned": ["...", "..."]
}
"""

with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    threads = json.load(f)

llm = OpenAI(model="gpt-4o", temperature=0.0)
summaries = []

for item in threads:
    thread_id = item["id"]
    thread_text = item["post"]
    print(f"Summarizing thread {thread_id}...")
    try:
        prompt = SUMMARIZE_PROMPT + f"\n\n<thread>\n{thread_text}\n</thread>"
        raw = str(llm.complete(prompt)).strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        result = json.loads(raw)
        summaries.append({
            "thread_id": thread_id,
            "summary": result.get("summary", ""),
            "functions_mentioned": result.get("functions_mentioned", [])
        })
    except Exception as e:
        print(f"ERROR: thread {thread_id}: {e}")
        summaries.append({
            "thread_id": thread_id,
            "summary": "",
            "functions_mentioned": []
        })

print(f"\nTotal summaries: {len(summaries)}")

with open("summaries.json", "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2)
print("Saved to summaries.json")

Summarizing thread 136...
Summarizing thread 138...
Summarizing thread 162...
Summarizing thread 203...
Summarizing thread 245...
Summarizing thread 346...
Summarizing thread 589...
Summarizing thread 773...
Summarizing thread 1168...
Summarizing thread 1272...
Summarizing thread 1436...
Summarizing thread 1459...
Summarizing thread 1522...
Summarizing thread 1724...
Summarizing thread 1905...
Summarizing thread 1935...
Summarizing thread 1947...
Summarizing thread 1976...
Summarizing thread 2245...
Summarizing thread 2459...
Summarizing thread 2656...
Summarizing thread 2708...
Summarizing thread 2714...
Summarizing thread 2718...
Summarizing thread 2742...
Summarizing thread 2747...
Summarizing thread 2840...
Summarizing thread 2926...
Summarizing thread 2962...
Summarizing thread 2992...
Summarizing thread 3021...
Summarizing thread 3112...
Summarizing thread 3209...
Summarizing thread 3260...
Summarizing thread 3303...
Summarizing thread 3418...
Summarizing thread 3422...
Summarizi

## Step 2. Chunk RST Sections

In [5]:
import os
import json
import re

CHUNK_DOCS_PATHS = [
    "../docs/nmodl",
    "../docs/progref",
]

MIN_CHUNK_CHARS = 50       # filter near-empty chunks
MAX_CHUNK_CHARS = 8000     # split large chunks at paragraph boundaries

# RST heading underline characters in conventional hierarchy order
HEADING_HIERARCHY = ['#', '*', '=', '-', '^', '"', '~', '+']

# RST directives that define a new documentable item
DIRECTIVE_PATTERN = re.compile(r'^\.\. (method|class|function|attribute|data):: (.+)$')

def get_heading_level(underline_char):
    """Map RST underline character to an integer heading level."""
    if underline_char in HEADING_HIERARCHY:
        return HEADING_HIERARCHY.index(underline_char) + 1
    return len(HEADING_HIERARCHY) + 1

def is_rst_heading(lines, line_idx):
    """Return True if lines[line_idx] is an RST section heading (underline style)."""
    if line_idx < 0 or line_idx + 1 >= len(lines):
        return False
    line = lines[line_idx]
    next_line = lines[line_idx + 1]
    # Exclude table rows and separators
    if line.strip().startswith('|') or line.strip().startswith('+'):
        return False
    return (
        len(line.strip()) > 0 and
        bool(re.match(r'^[=\-~^#*+`\'"_]{3,}$', next_line.strip()))
    )

def split_large_chunk(chunk):
    """
    Split a chunk whose content exceeds MAX_CHUNK_CHARS into smaller pieces,
    breaking at paragraph boundaries (double newlines). Returns a list of chunks.
    If the chunk is within the limit, returns a single-element list.
    """
    content = chunk["content"]
    if len(content) <= MAX_CHUNK_CHARS:
        return [chunk]

    paragraphs = re.split(r'\n{2,}', content)
    sub_chunks = []
    current_parts = []
    current_len = 0
    part_num = 1

    for para in paragraphs:
        para_len = len(para) + 2  # account for the newline separator
        if current_len + para_len > MAX_CHUNK_CHARS and current_parts:
            sub_chunks.append({**chunk,
                "section_heading": f"{chunk['section_heading']} (part {part_num})",
                "content": "\n\n".join(current_parts)
            })
            part_num += 1
            current_parts = [para]
            current_len = para_len
        else:
            current_parts.append(para)
            current_len += para_len

    if current_parts:
        label = f"{chunk['section_heading']} (part {part_num})" if part_num > 1 else chunk['section_heading']
        sub_chunks.append({**chunk, "section_heading": label, "content": "\n\n".join(current_parts)})

    return sub_chunks

def chunk_rst_file(file_path):
    """Split an RST file into sections by heading or directive."""
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    # Find all split points: (line_idx, heading_text, heading_level)
    split_points = []
    for i in range(len(lines)):
        # Check for underline-style heading
        if is_rst_heading(lines, i):
            underline_char = lines[i + 1][0] if lines[i + 1].strip() else '='
            split_points.append((i, lines[i].strip(), get_heading_level(underline_char)))
        else:
            # Check for directive-style heading (.. method::, .. class::, etc.)
            m = DIRECTIVE_PATTERN.match(lines[i].strip())
            if m:
                split_points.append((i, m.group(2).strip(), 9))  # level 9 = sub-item

    if not split_points:
        content = '\n'.join(lines).strip()
        if not content:
            return []
        return [{
            "file_path": file_path,
            "section_heading": os.path.basename(file_path),
            "heading_level": 1,
            "content": content
        }]

    chunks = []
    for idx, (line_idx, heading_text, heading_level) in enumerate(split_points):
        section_start = line_idx + 2 if is_rst_heading(lines, line_idx) else line_idx + 1
        section_end = split_points[idx + 1][0] if idx + 1 < len(split_points) else len(lines)
        content = '\n'.join(lines[section_start:section_end]).strip()
        if not content:
            continue
        chunks.append({
            "file_path": file_path,
            "section_heading": heading_text,
            "heading_level": heading_level,
            "content": content
        })

    return chunks


# Walk docs directories and chunk all RST files
raw_chunks = []
for docs_path in CHUNK_DOCS_PATHS:
    if not os.path.exists(docs_path):
        print(f"WARNING: {docs_path} does not exist, skipping.")
        continue
    for root, dirs, files in os.walk(docs_path):
        for fname in sorted(files):
            if fname.endswith('.rst'):
                file_path = os.path.join(root, fname)
                try:
                    raw_chunks.extend(chunk_rst_file(file_path))
                except Exception as e:
                    print(f"ERROR: {file_path}: {e}")

print(f"Raw chunks: {len(raw_chunks)}")

# --- Post-processing ---

# 1. Merge duplicate headings (same file + same heading) — e.g. HOC/Python tab splits
merged = {}
for c in raw_chunks:
    key = (c["file_path"], c["section_heading"])
    if key in merged:
        merged[key]["content"] += "\n\n" + c["content"]
    else:
        merged[key] = dict(c)

deduped = list(merged.values())
print(f"After merging duplicates: {len(deduped)} (removed {len(raw_chunks) - len(deduped)})")

# 2. Filter near-empty chunks
filtered = [c for c in deduped if len(c["content"].strip()) >= MIN_CHUNK_CHARS]
print(f"After filtering near-empty (<{MIN_CHUNK_CHARS} chars): {len(filtered)} (removed {len(deduped) - len(filtered)})")

# 3. Split large chunks at paragraph boundaries
split_chunks = []
split_count = 0
for c in filtered:
    parts = split_large_chunk(c)
    if len(parts) > 1:
        split_count += 1
    split_chunks.extend(parts)
print(f"After splitting large (>{MAX_CHUNK_CHARS} chars): {len(split_chunks)} chunks ({split_count} chunks split)")

# 4. Final filter — remove any near-empty chunks created by splitting
all_chunks = [c for c in split_chunks if len(c["content"].strip()) >= MIN_CHUNK_CHARS]
if len(all_chunks) < len(split_chunks):
    print(f"After final filter: {len(all_chunks)} (removed {len(split_chunks) - len(all_chunks)} post-split empties)")

# Sanity check
still_large = [c for c in all_chunks if len(c["content"]) > MAX_CHUNK_CHARS]
still_small = [c for c in all_chunks if len(c["content"].strip()) < MIN_CHUNK_CHARS]
print(f"\nSanity check: {len(still_large)} chunks still >{MAX_CHUNK_CHARS} chars (single oversized paragraphs, unavoidable)")
print(f"Sanity check: {len(still_small)} chunks still <{MIN_CHUNK_CHARS} chars")

# Save to chunks.json
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)
print(f"\nSaved {len(all_chunks)} chunks to chunks.json")

Raw chunks: 1420
After merging duplicates: 1415 (removed 5)
After filtering near-empty (<50 chars): 1388 (removed 27)
After splitting large (>8000 chars): 1424 chunks (29 chunks split)
After final filter: 1423 (removed 1 post-split empties)

Sanity check: 5 chunks still >8000 chars (single oversized paragraphs, unavoidable)
Sanity check: 0 chunks still <50 chars

Saved 1423 chunks to chunks.json


## RAG 3. Build (or load) the index

In [5]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Configure models
Settings.llm = OpenAI(model="gpt-4o", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

if os.path.exists(INDEX_STORE_PATH):
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_STORE_PATH)
    index = load_index_from_storage(storage_context)
else:
    print("Building index from docs (this may take a few minutes)...")
    all_documents = []
    for docs_path in DOCS_PATHS:
        docs = SimpleDirectoryReader(
            input_dir=docs_path,
            recursive=True,
            required_exts=[".rst"],
        ).load_data()
        print(f"Loaded {len(docs)} .rst files from {docs_path}")
        all_documents.extend(docs)
    print(f"Total: {len(all_documents)} .rst files")
    index = VectorStoreIndex.from_documents(all_documents)
    index.storage_context.persist(persist_dir=INDEX_STORE_PATH)
    print("Index built and saved.")

Loading existing index...


## RAG 4. Define the prompt template

In [6]:
PROMPT_TEMPLATE = """
You are a technical documentation assistant helping integrate community
Q&A content into the NEURON simulator's official documentation.

Below are the most relevant excerpts from the current NEURON documentation,
each labelled with its source .rst file path:

---------------------
{context_str}
---------------------

Here is a forum Q&A thread that contains information to be integrated:

<thread>
{query_str}
</thread>

Instructions:
- Identify every function, method, or class that the thread adds new information about.
- For each one, produce a unified diff showing what should be added to the relevant .rst file.
- Use standard unified diff format:
    --- a/<rst_file_path>
    +++ b/<rst_file_path>
    @@ -<line>,<count> +<line>,<count> @@
     (context lines with a leading space)
    +(new lines with a leading +)
- Base diffs on the existing documentation excerpts shown above.
- New content must match RST style (directives, inline code, section structure).
- Only include a code block if you can provide a complete, working example — either from the thread or generated yourself. Never include a partial or placeholder code block.
- Use ``.. code-block:: python`` only for actual Python code. Use ``.. code-block:: none`` for HOC code or any other language.
- Where relevant, provide both a HOC and a Python example to support users of both interfaces.
- If multiple files need changes, concatenate their diffs.
- Never insert content in the middle of a function definition, parameter description, or syntax block. Only add content at the end of the most relevant section, or after the last related paragraph.

Return ONLY the unified diff text, with no preamble, explanation, or code fences.
Only include information clearly supported by the thread. Do not extrapolate.
"""

## RAG 5. Run a query for a single forum thread

In [7]:
import json
# Paste your forum thread here
with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    entry = json.load(f)

In [ ]:
from llama_index.llms.openai import OpenAI
import re

def is_rst_heading(lines, line_idx):
    """Return True if lines[line_idx] is an RST section heading."""
    if line_idx < 0 or line_idx + 1 >= len(lines):
        return False
    line = lines[line_idx]
    next_line = lines[line_idx + 1]
    return (
        len(line.strip()) > 0 and
        bool(re.match(r'^[=\-~^#*+`\'"_]{3,}$', next_line.strip()))
    )

def extract_section(file_path, chunk_text, max_lines=150):
    """
    Read file_path and return the RST section containing chunk_text.
    Falls back to chunk_text if the file cannot be read or chunk is not found.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            full_text = f.read()
    except OSError:
        return chunk_text

    lines = full_text.split('\n')
    search_text = chunk_text[:120].strip()
    chunk_pos = full_text.find(search_text)
    if chunk_pos == -1:
        return chunk_text

    chunk_line = full_text[:chunk_pos].count('\n')

    # Walk upward to find the nearest heading
    section_start = max(0, chunk_line - max_lines)
    for i in range(chunk_line, -1, -1):
        if is_rst_heading(lines, i):
            section_start = i
            break

    # Walk downward to find the next heading
    section_end = min(len(lines), chunk_line + max_lines)
    for i in range(chunk_line + 1, len(lines)):
        if is_rst_heading(lines, i):
            section_end = i
            break

    # Cap to max_lines to avoid sending huge sections
    if section_end - section_start > max_lines:
        section_end = section_start + max_lines

    return '\n'.join(lines[section_start:section_end])


def process_thread(thread_id, thread_text):
    """Query the index with a forum thread and return a unified diff string."""
    estimated_tokens = len(thread_text) // 4
    if estimated_tokens > 8000:
        print(f"WARNING: Thread {thread_id} is large (~{estimated_tokens} tokens) and may hit the token limit.")
    try:
        # Step 1: retrieve the most relevant chunks
        retriever = index.as_retriever(similarity_top_k=5)
        nodes = retriever.retrieve(thread_text)

        # Step 2: expand each chunk to its full RST section
        context_parts = []
        for node in nodes:
            file_path = node.metadata.get('file_path', '')
            chunk_text = node.get_content()
            section = extract_section(file_path, chunk_text)
            context_parts.append(f"Source: {file_path}\n\n{section}")

        context_str = "\n\n---\n\n".join(context_parts)

        # Step 3: fill the prompt and call GPT directly
        prompt = PROMPT_TEMPLATE.format(context_str=context_str, query_str=thread_text)
        llm = OpenAI(model="gpt-4o", temperature=0.2)
        raw = str(llm.complete(prompt)).strip()

        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        return raw

    except Exception as e:
        error_msg = str(e)
        if "maximum context length" in error_msg or "token" in error_msg.lower():
            print(f"ERROR: Thread {thread_id} exceeded token limit (~{estimated_tokens} tokens).")
        else:
            print(f"ERROR: Thread {thread_id} failed: {error_msg}")
        return None


# Test on the first thread
forum_id = entry[1]["id"]
forum_thread = entry[1]["post"]

result = process_thread(forum_id, forum_thread)
print(result)

## RAG 6. Bulk processing

In [ ]:
import os

OUTPUT_DIR = "doc-project/diffs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for item in entry:
    thread_id = item['id']
    thread_text = item['post']
    print(f"Processing thread {thread_id}...")
    diff = process_thread(thread_id, thread_text)
    if diff:
        out_path = os.path.join(OUTPUT_DIR, f"thread_{thread_id}.diff")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(diff)
        print(f"  Saved: {out_path}")
    else:
        print(f"  Skipped thread {thread_id} (error or empty response).")

## Step 3. Evaluate with DeepEval

In [ ]:
import csv
import json
import os
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRelevancyMetric, AnswerRelevancyMetric

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# Load summaries keyed by thread_id
with open("summaries.json") as f:
    summaries = {s["thread_id"]: s["summary"] for s in json.load(f)}

with open("d_lst_posts.json") as f:
    threads = json.load(f)

# GPT-4o as judge — uses OPENAI_API_KEY already loaded from .env
JUDGE_MODEL = "gpt-4o"
cr_metric = ContextualRelevancyMetric(model=JUDGE_MODEL, include_reason=False)
ar_metric = AnswerRelevancyMetric(model=JUDGE_MODEL, include_reason=False)

retriever = index.as_retriever(similarity_top_k=5)
results = []

for item in threads:
    thread_id   = item["id"]
    thread_text = item["post"]
    summary     = summaries.get(thread_id, "")

    if not summary:
        print(f"[{thread_id}] no summary — skipping")
        continue

    print(f"Evaluating thread {thread_id}...")

    # Retrieve top-5 chunks — same settings as process_thread (deterministic)
    try:
        nodes = retriever.retrieve(thread_text)
        retrieval_context = [node.get_content() for node in nodes]
    except Exception as e:
        print(f"  retrieval error: {e}")
        continue

    # Generate the diff
    diff = process_thread(thread_id, thread_text)
    if not diff:
        print(f"  no diff — skipping")
        continue

    # LLMTestCase fields:
    #   input             = thread summary (clean query for the RAG)
    #   actual_output     = generated diff (what the pipeline produced)
    #   retrieval_context = top-5 RST chunks retrieved from the index
    test_case = LLMTestCase(
        input=summary,
        actual_output=diff,
        retrieval_context=retrieval_context,
    )

    row = {"thread_id": thread_id, "contextual_relevancy": None, "answer_relevancy": None}

    for metric, key in [(cr_metric, "contextual_relevancy"), (ar_metric, "answer_relevancy")]:
        try:
            metric.measure(test_case)
            row[key] = round(metric.score, 4)
        except Exception as e:
            print(f"  {key} error: {e}")

    results.append(row)
    print(f"  CR={row['contextual_relevancy']}  AR={row['answer_relevancy']}")

# Write CSV
out_path = "eval_results.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["thread_id", "contextual_relevancy", "answer_relevancy"])
    writer.writeheader()
    writer.writerows(results)

print(f"
Saved {len(results)} rows → {out_path}")

## RAG 7. Save results to a file for review

In [ ]:
# Diff files are saved to doc-project/diffs/thread_<id>.diff during bulk processing above.
# To apply a diff: patch -p1 < doc-project/diffs/thread_<id>.diff